In [1]:
import caveclient
import pandas as pd
import numpy as np
import os
import tqdm
import standard_transform
from pathlib import Path

from joblib import Parallel, delayed

/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.2) or chardet (7.5.1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [2]:
## for temp notetaking 
from IPython.display import HTML
HTML("""
<style>
.yynote { color: red; font-weight: bold; font-size: 12pt; }
</style>
""")

<span class="yynote">I will leave comments</span>

# Load data from CAVE

In [5]:
trafo_streamline = standard_transform.datasets.v1dd_streamline_nm()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)

cave_token = os.environ["CUSTOM_KEY"]
mat_version = 1196

/tmp/ipykernel_125/4063311819.py:1: DeprecationWarning: v1dd_streamline_nm(...) is deprecated and will be removed in a future release; use v1dd_streamline(resolution='nm', ...) instead.
  trafo_streamline = standard_transform.datasets.v1dd_streamline_nm()


In [6]:
client = caveclient.CAVEclient(
    "v1dd",
    server_address="https://global.em.brain.allentech.org",
    auth_token=cave_token,
    version=mat_version,
)

Updated datastack-to-server cache — 'https://global.em.brain.allentech.org' will now be used by default for datastack 'v1dd'


## Proofreading and data quality

Understanding this variablity in data quality is critical when interpretting electron microscopy reconstructions.

Automated segmentation of neuronal processes in dense EM imaging is challenging at the size of entire neurons, which can have millimeters of axons and dendrites. The automated segmentation algorithms used in the EM data for this project are not perfect, and so proofreading is necessary to obtain accurate reconstructions of a cell and confidence in the connectivity

Axon and dendrite compartment status are marked separately, as proofreading effort was applied differently to the different compartments in some cells.  In all cases, a status of `TRUE` indicates that false merges have been comprehensively removed, and the compartment is at least ‘clean’. Consult the ‘strategy’ column if completeness of the compartment is relevant to your  research.

Some cells were extended to different degrees of completeness, or with different research goals in mind. This is denoted by 'strategy_axon', which may be one of:

<ul>
    <li>none: No cleaning, and no extension, and status is `FALSE`. </li>
    <li>axon_partially_extended: The axon was extended outward from the soma, following each branch to its termination. Output synapses represent a sampling of potential partners. </li>
    <li>axon_interareal: The axon was extended with a preference for branches that projected to other brain areas. Some axon branches were fully extended, but local connections may be incomplete. Output synapses represent a sampling of potential partners. </li>
    <li>axon_fully_extended: Axon was extended outward from the soma, following each branch to its termination. After initial extension, every endpoint was identified, manually inspected, and extended again if possible. Output synapses represent a largely complete sampling of partners.. </li>
</ul>

<b> For this workshop, we treat all cells with at least `axon_partially_extended` as equally trustworth.</b> This may not be a safe assumption for all analysis, and we are happy to provide more guidance depending on the research question.

<span class="yynote">the md above says "partilally extended" but the code below saves "fully extended"</span>

In [7]:
neuron_soma_df = client.materialize.query_table("neurons_soma_model")
single_soma_df = client.materialize.query_view(
    "single_somas", split_positions=True, desired_resolution=[1, 1, 1]
)
axon_proof_df = client.materialize.query_table(
    "proofreading_status_and_strategy",
    filter_in_dict={"strategy_axon": ["axon_fully_extended"]},
)

# get root ids
dendrite_proof_root_ids = np.array(single_soma_df["pt_root_id"])
dendrite_proof_root_ids = dendrite_proof_root_ids[
    np.isin(dendrite_proof_root_ids, neuron_soma_df["pt_root_id"])
]
axon_proof_root_ids = np.array(axon_proof_df["pt_root_id"])

In [8]:
dendrite_proof_root_ids

array([864691132496108732, 864691132511800666, 864691132525163794, ...,
       864691133314213712, 864691133314222928, 864691133314231632],
      shape=(63986,))

In [9]:
axon_proof_root_ids

array([864691132534275418, 864691132534315610, 864691132535664474, ...,
       864691132549572162, 864691132721794547, 864691132624685004],
      shape=(1210,))

## Cell information

<span class="yynote">why clean up L23, L1inh, L6b cell types?</span>

In [10]:
cell_type_df = client.materialize.query_table(
    "cell_type_snds", desired_resolution=[1, 1, 1], split_positions=True
).rename(columns={"classification_system": "cell_type_coarse"})

cell_type_df = cell_type_df[
    ~cell_type_df["cell_type"].map(
        lambda x: x.startswith("L23_")
        or x.startswith("L1_inh")
        or x.startswith("L6_6b")
    )
]

nuc_df = client.materialize.query_view(
    "nucleus_alternative_lookup", split_positions=True, desired_resolution=[1, 1, 1]
)

reference_point = np.array([900_000.0, 700_000.0, 300_000.0])
nuc_df[["pt_position_trform_x", "pt_position_trform_y", "pt_position_trform_z"]] = trafo_streamline.radial_points(reference_point, nuc_df[["pt_position_x", "pt_position_y", "pt_position_z"]]) #* 1000 leave in microns

In [11]:
def map_cell_type(ct_str):
    if ct_str.startswith("L"):
        return "-".join(ct_str.split("_")[:2])
    else:
        return ct_str[:3]


cell_type_df["cell_type"] = cell_type_df["cell_type"].map(map_cell_type)
cell_type_df["cell_type_coarse"] = cell_type_df["cell_type_coarse"].map(
    lambda x: {"inh": "I", "exc": "E"}[x]
)

In [12]:
soma_ct_df = pd.merge(
    nuc_df[
        [
            "id",
            "pt_position_x",
            "pt_position_y",
            "pt_position_z",
            "pt_position_trform_x",
            "pt_position_trform_y",
            "pt_position_trform_z",
            "pt_root_id",
            "volume",
        ]
    ],
    cell_type_df[["pt_root_id", "cell_type_coarse", "cell_type"]],
    how="left",
    on="pt_root_id",
)

In [13]:
soma_ct_df

,id,pt_position_x,pt_position_y,pt_position_z,pt_position_trform_x,pt_position_trform_y,pt_position_trform_z,pt_root_id,volume,cell_type_coarse,cell_type
0,228132,632828,749849,738270,-313.710105,563.588881,398.097352,864691132737039043,458.464831,NaN,NaN
1,543247,1304922,977915,83880,343.043701,607.352314,-298.258541,864691132730839988,73.34594,NaN,NaN
2,203262,624680,531094,283770,-248.956909,222.054907,16.022035,864691132654552792,338.276613,E,L3-IT
3,350562,894573,478559,163530,30.864259,135.367984,-105.097754,864691132773514104,326.9654,E,L2-IT
4,718122,1729859,674111,781200,804.489146,493.264462,468.374144,864691132774106773,333.888647,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
207450,527607,1262940,628094,734445,352.445121,436.611300,437.410164,864691132639606383,100.547645,NaN,NaN
207451,168582,491518,1057067,92070,-504.222046,696.211864,-325.065319,864691133042980384,369.919126,NaN,NaN
207452,29422,302330,415005,81855,-549.261536,52.309023,-185.186054,0,285.031368,NaN,NaN
207453,422767,1065603,538932,36405,198.546576,159.482968,-238.086820,864691132851361283,394.72429,NaN,NaN


In [14]:
# Quantify and remove ambiguous nucleus-id <-> pt_root_id mappings.
# Two kinds of ambiguity break the assumed 1:1 nucleus(id) <-> segment(pt_root_id) mapping:
#   (a) one id       -> multiple pt_root_ids
#   (b) multiple ids -> one pt_root_id
# Use unique (id, pt_root_id) pairs so duplicate rows introduced by the left-merge with
# cell_type_df are not miscounted as mapping ambiguity.
id_root_pairs = soma_ct_df[["id", "pt_root_id"]].drop_duplicates()

root_ids_per_id = id_root_pairs.groupby("id")["pt_root_id"].nunique()
ids_per_root_id = id_root_pairs.groupby("pt_root_id")["id"].nunique()

ambiguous_ids = root_ids_per_id.index[root_ids_per_id > 1]
ambiguous_root_ids = ids_per_root_id.index[ids_per_root_id > 1]

print(f"ids mapping to >1 pt_root_id (one id -> many root_ids): {len(ambiguous_ids)}")
print(f"pt_root_ids mapping to >1 id (many ids -> one root_id): {len(ambiguous_root_ids)}")

ids mapping to >1 pt_root_id (one id -> many root_ids): 0
pt_root_ids mapping to >1 id (many ids -> one root_id): 19616


In [15]:
# Drop every row participating in either kind of ambiguity, keeping only clean 1:1 rows.
ambiguous_mask = soma_ct_df["id"].isin(ambiguous_ids) | soma_ct_df[
    "pt_root_id"
].isin(ambiguous_root_ids)
print(
    f"rows dropped due to ambiguous mapping: {int(ambiguous_mask.sum())} / {len(soma_ct_df)}"
)

soma_ct_unique_df = soma_ct_df[~ambiguous_mask].reset_index(drop=True)
print(f"soma_ct_unique_df rows: {len(soma_ct_unique_df)}")

rows dropped due to ambiguous mapping: 64007 / 207455
soma_ct_unique_df rows: 143448


In [16]:
# Restrict proofread dendrite / axon root-id lists to segments still present in the
# cleaned soma table, and quantify the overlap.
unique_root_ids = np.array(sorted(set(soma_ct_unique_df["pt_root_id"])))

dendrite_kept_mask = np.isin(dendrite_proof_root_ids, unique_root_ids)
axon_kept_mask = np.isin(axon_proof_root_ids, unique_root_ids)

print(
    f"proofread dendrites kept (in cleaned soma table): "
    f"{int(dendrite_kept_mask.sum())} / {len(dendrite_proof_root_ids)}"
)
print(
    f"proofread axons kept (in cleaned soma table): "
    f"{int(axon_kept_mask.sum())} / {len(axon_proof_root_ids)}"
)

dendrite_proof_root_ids = dendrite_proof_root_ids[dendrite_kept_mask]
axon_proof_root_ids = axon_proof_root_ids[axon_kept_mask]

proofread dendrites kept (in cleaned soma table): 63986 / 63986
proofread axons kept (in cleaned soma table): 1164 / 1210


## Synapses

<span class="yynote">works but takes too long, i used the saved version</span>

In [17]:
DATA_ROOT = Path("/data/v1dd_1196")
syn_df = pd.read_feather(DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather")
syn_label_df = pd.read_feather(DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather")

In [19]:
syn_df.head()

,id,pre_pt_position_x,pre_pt_position_y,pre_pt_position_z,post_pt_position_x,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,pre_pt_root_id,post_pt_root_id
0,354386968,758200.5,802316.1,304380.0,757861.0,802558.6,304650.0,757967.7,802597.4,304380.0,240,864691132536286810,864691132734919083
1,378070488,792063.2,514342.5,183735.0,792664.6,514284.3,183915.0,792412.4,514294.0,183735.0,3056,864691132572190492,864691132606767301
2,499493001,977071.3,390075.8,191340.0,976974.3,390104.9,190935.0,976838.5,390337.7,190935.0,1346,864691132573738810,864691132747578447
3,119675985,444260.0,602544.6,3285.0,443988.4,602311.8,3555.0,444182.4,602370.0,3780.0,3637,864691132572564252,864691132654028028
4,220616943,574501.9,337249.6,258570.0,574152.7,337016.8,258570.0,574337.0,336900.4,258570.0,420,864691132558380553,864691132828255906


## Synapse types

In [20]:
syn_label_df.head()

,tag
id,
35903092,spine
51716092,spine
59062330,spine
62437192,shaft
62841618,spine


# Save data to `CommonConnectivityMatrix` format

In [21]:
from connects_common_connectivity.models import (
    AlgorithmRun,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Modality,
    Unit,
)
from connects_common_connectivity.io import write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema

from pathlib import Path

import pyarrow as pa
from deltalake import write_deltalake



In [22]:
OUTPUT_ROOT = "../results/v1dd_1196_v3/"
PROJECT_ID  = "v1dd"
RELEASE     = "1196"

## `Datasets`

In [23]:
V1DD_PUBLICATION = "https://github.com/AllenInstitute/v1dd_physiology"
DATASET_EM   = "v1dd_1196_em"
DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
DATASET_PROOFREAD_DEND = "v1dd_1196_proofread_dendrites"

datasets = [
    DataSet(
        id=DATASET_EM,
        name="V1DD release 1196 — EM somas",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_AXON,
        name="V1DD release 1196 — proofread axons cohort",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_DEND,
        name="V1DD release 1196 — proofread dendrites cohort",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
]

result = write_models(datasets, output_root=OUTPUT_ROOT)

## `DataItem`s and `DataItemDataSetAssociation`s
DataItems: one per cleaned soma segment.
- id   = pt_root_id (segment / EM reconstruction identifier)
- name = nucleus id (soma_ct_unique_df["id"])

DataItemDataSetAssociations: link each DataItem to the dataset cohort(s) it belongs to.

In [24]:
dataitems = [
    DataItem(
        id=str(row.pt_root_id),
        name=str(row.id),
        project_id=PROJECT_ID,
    )
    for row in soma_ct_unique_df.itertuples()
]

result = write_models(dataitems, output_root=OUTPUT_ROOT)
print(f"DataItem rows written: {result.rows_written} (total in batch: {len(dataitems)})")

# Proofread cohorts were filtered to soma_ct_unique_df upstream, so every proofread
# root id must already have a DataItem.
dataitem_ids = {item.id for item in dataitems}
assert all(str(r) in dataitem_ids for r in dendrite_proof_root_ids), "dendrite proof root id missing a DataItem"
assert all(str(r) in dataitem_ids for r in axon_proof_root_ids), "axon proof root id missing a DataItem"

DataItem rows written: 143448 (total in batch: 143448)


In [29]:
associations = []

# EM soma cohort: every cleaned soma.
associations += [
    DataItemDataSetAssociation(
        dataitem_id=str(root_id),
        dataset_id=DATASET_EM,
        project_id=PROJECT_ID,
    )
    for root_id in unique_root_ids
]

# Proofread dendrite cohort.
associations += [
    DataItemDataSetAssociation(
        dataitem_id=str(root_id),
        dataset_id=DATASET_PROOFREAD_DEND,
        project_id=PROJECT_ID,
    )
    for root_id in dendrite_proof_root_ids
]

# Proofread axon cohort.
associations += [
    DataItemDataSetAssociation(
        dataitem_id=str(root_id),
        dataset_id=DATASET_PROOFREAD_AXON,
        project_id=PROJECT_ID,
    )
    for root_id in axon_proof_root_ids
]

result = write_models(associations, output_root=OUTPUT_ROOT)
print(f"DataItemDataSetAssociation rows written: {result.rows_written} (total in batch: {len(associations)})")

DataItemDataSetAssociation rows written: 208598 (total in batch: 208598)


# `CellFeatureSet` , `CellFeatureDefinition`, `CellFeatureMatrix`
For soma voxel and transformed coordinates and soma volume

<span class="yynote">double check feature descriptions and units</span>

In [30]:
FEATURE_SET_ID = "v1dd_soma_spatial"
output_root = Path(OUTPUT_ROOT)

In [31]:
soma_ct_unique_df.head()

,id,pt_position_x,pt_position_y,pt_position_z,pt_position_trform_x,pt_position_trform_y,pt_position_trform_z,pt_root_id,volume,cell_type_coarse,cell_type
0,203262,624680,531094,283770,-248.956909,222.054907,16.022035,864691132654552792,338.276613,E,L3-IT
1,350562,894573,478559,163530,30.864259,135.367984,-105.097754,864691132773514104,326.9654,E,L2-IT
2,718122,1729859,674111,781200,804.489146,493.264462,468.374144,864691132774106773,333.888647,NaN,NaN
3,680726,1640231,677370,768015,714.854245,493.323720,454.734800,864691132780109973,706.847075,NaN,NaN
4,544582,1316562,974035,157590,348.218706,624.536270,-228.775916,864691132736203675,200.069363,NaN,NaN


In [32]:
# CellFeatureDefinitions: soma voxel coords, transformed coords, and volume.
soma_fds = [
    CellFeatureDefinition(
        id="soma_voxel_x",
        description="Soma nucleus centroid x in EM voxel coordinates.",
        unit=Unit.ARBITRARY_UNIT.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_voxel_y",
        description="Soma nucleus centroid y in EM voxel coordinates.",
        unit=Unit.ARBITRARY_UNIT.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_voxel_z",
        description="Soma nucleus centroid z in EM voxel coordinates.",
        unit=Unit.ARBITRARY_UNIT.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_transformed_x",
        description="Soma nucleus centroid x in streamline-transformed cortical coordinates.",
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_transformed_y",
        description="Soma nucleus centroid y in streamline-transformed cortical coordinates.",
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_transformed_z",
        description="Soma nucleus centroid z in streamline-transformed cortical coordinates.",
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
    CellFeatureDefinition(
        id="soma_volume",
        description="Nucleus volume.",
        unit=Unit.MICRONS_CUBED.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    ),
]
result = write_models(soma_fds, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition rows written: {result.rows_written} (total in batch: {len(soma_fds)})")

CellFeatureDefinition rows written: 7 (total in batch: 7)


In [33]:
# CellFeatureSet.
soma_fs = CellFeatureSet(
    id=FEATURE_SET_ID,
    description=(
        "Soma spatial features for V1DD release 1196: nucleus centroid in EM voxel "
        "coordinates and in streamline-transformed cortical coordinates, plus nucleus volume."
    ),
    feature_definition_ids=[fd.id for fd in soma_fds],
    extraction_method=(
        "Queried from CAVE nucleus tables; cortical coordinates via "
        "standard_transform v1dd_streamline_nm (nm converted to microns)."
    ),
    project_id=PROJECT_ID,
)
result = write_models([soma_fs], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet rows written: {result.rows_written}")

CellFeatureSet rows written: 1


In [34]:
# Wide-form feature matrix, one row per DataItem (root id).
# Transformed coordinates are in nm -> divide by 1000 for microns.
soma_feat_wide = pd.DataFrame(
    {
        "id": soma_ct_unique_df["pt_root_id"].astype(str).values,
        "soma_voxel_x": soma_ct_unique_df["pt_position_x"].astype("float32").values,
        "soma_voxel_y": soma_ct_unique_df["pt_position_y"].astype("float32").values,
        "soma_voxel_z": soma_ct_unique_df["pt_position_z"].astype("float32").values,
        "soma_transformed_x": (soma_ct_unique_df["pt_position_trform_x"] / 1000).astype("float32").values,
        "soma_transformed_y": (soma_ct_unique_df["pt_position_trform_y"] / 1000).astype("float32").values,
        "soma_transformed_z": (soma_ct_unique_df["pt_position_trform_z"] / 1000).astype("float32").values,
        "soma_volume": soma_ct_unique_df["volume"].astype("float32").values,
    }
)
soma_feat_wide["project_id"] = PROJECT_ID
soma_feat_wide["feature_set_id"] = FEATURE_SET_ID

soma_schema = build_cell_feature_matrix_schema(soma_fs, soma_fds, cell_index_column="id")
soma_table = pa.Table.from_pandas(soma_feat_wide, schema=soma_schema, preserve_index=False)
write_deltalake(
    output_root / f"cellfeatures/{FEATURE_SET_ID}",
    soma_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Wide soma feature matrix written:", soma_table.shape)

# CellFeatureMatrix pointer.
soma_cfm = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{output_root.resolve()}/cellfeatures/{FEATURE_SET_ID}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([soma_cfm], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix rows written: {result.rows_written}")

Wide soma feature matrix written: (143448, 10)
CellFeatureMatrix rows written: 1


# `ClusterHierarchy` and `Clustermembership`
for cell types

In [35]:
HIERARCHY_ID = "v1dd_cell_types"
RUN_ID = "v1dd_cell_type_annotation"
ROOT_ID = "neuron"

In [36]:
# Cells with a cell-type annotation, one row per soma segment (DataItem id = pt_root_id).
n_soma = soma_ct_unique_df.shape[0]
ct_df = soma_ct_unique_df.dropna(subset=["cell_type_coarse", "cell_type"])
print(f"annotated somas: {len(ct_df)} / {n_soma}")

# fine cell_type -> coarse (E/I); must be consistent.
fine_to_coarse = ct_df.groupby("cell_type")["cell_type_coarse"].unique()
assert all(len(v) == 1 for v in fine_to_coarse), "a fine cell_type maps to multiple coarse classes"
fine_to_coarse = {k: v[0] for k, v in fine_to_coarse.items()}

coarse_ids = sorted(ct_df["cell_type_coarse"].unique())
fine_ids = sorted(ct_df["cell_type"].unique())

annotated somas: 49192 / 143448


In [37]:
fine_to_coarse

{'DTC': 'I',
 'ITC': 'I',
 'L2-IT': 'E',
 'L3-IT': 'E',
 'L4-IT': 'E',
 'L5-ET': 'E',
 'L5-IT': 'E',
 'L5-NP': 'E',
 'L6-CT': 'E',
 'L6-IT': 'E',
 'PTC': 'I',
 'STC': 'I'}

In [38]:
# Cluster tree: root (level 0) -> coarse E/I (level 1) -> fine cell_type (level 2).
clusters = [
    Cluster(id=ROOT_ID, hierarchy_id=HIERARCHY_ID, parent=None, children=coarse_ids, level=0)
]
for c in coarse_ids:
    children = sorted(f for f in fine_ids if fine_to_coarse[f] == c)
    clusters.append(
        Cluster(id=c, hierarchy_id=HIERARCHY_ID, parent=ROOT_ID, children=children, level=1)
    )
for f in fine_ids:
    clusters.append(
        Cluster(id=f, hierarchy_id=HIERARCHY_ID, parent=fine_to_coarse[f], children=None, level=2)
    )

result = write_models(clusters, output_root=OUTPUT_ROOT)
print(f"Cluster rows written: {result.rows_written} "
      f"(root + {len(coarse_ids)} coarse + {len(fine_ids)} fine = {len(clusters)})")

Cluster rows written: 15 (root + 2 coarse + 12 fine = 15)


In [39]:
# Provenance AlgorithmRun: cell types are CAVE annotations, not a clustering run in this notebook.
run_row = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="V1DD cell-type annotation (CAVE table cell_type_snds)",
    algorithm_version=str(RELEASE),
    input_dataset=DATASET_EM,
    json_object=None,
    run_timestamp=None,
    score_description=None,
    distance_description=None,
)
result = write_models([run_row], output_root=OUTPUT_ROOT)
print(f"AlgorithmRun rows written: {result.rows_written}")

AlgorithmRun rows written: 1


In [40]:
# ClusterHierarchy linking the run and listing all clusters.
hierarchy_row = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[c.id for c in clusters],
)
result = write_models([hierarchy_row], output_root=OUTPUT_ROOT)
print(f"ClusterHierarchy rows written: {result.rows_written}")

ClusterHierarchy rows written: 1


In [41]:
# ClusterMembership: one row per (cell x ancestor) - propagate fine -> coarse -> root.
memberships = []
for row in ct_df.itertuples():
    item = str(row.pt_root_id)
    for cluster_id in (row.cell_type, row.cell_type_coarse, ROOT_ID):
        memberships.append(
            ClusterMembership(
                item=item,
                cluster=cluster_id,
                hierarchy_id=HIERARCHY_ID,
                probability=1.0,
                project_id=PROJECT_ID,
            )
        )
result = write_models(memberships, output_root=OUTPUT_ROOT)
print(f"ClusterMembership rows written: {result.rows_written} ({len(ct_df)} cells x 3 levels)")

ClusterMembership rows written: 147576 (49192 cells x 3 levels)
